# Get BDB2026 Player-Tracking Data for WRs

For all of our 2023 rookie WRs, we want to get tracking data for their player plays from the 2023 season using BDB2026 data. 

A very important piece of what we're doing here is ***determining who the CB is that's guarding them***. 

Since we want to find separation during man coverage situations, we will want to find who the CB is that's guarding them. Currently, the BDB2026 data has tracking data for all defenders on a given player. We want to determine which CB is guarding our WR of interest.

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

In [ ]:
# ======================================
# WR Route Player-Tracking Data
# ======================================

bdb26_df = (
    pl.read_csv(str(SEPARATION_DATA_DIR / "00_2023_Rookie_Player_Time_Level_Play_Details_Data.csv"))
    # -----------------------
    # Only Keep WRs
    # -----------------------
    .filter(pl.col("player_position") == "WR")
    # -----------------------
    # Create Route Variable
    # -----------------------
    .with_columns(
        route = (
            # If target and has route_of_targeted_receiver data, then use that
            pl.when((pl.col("target") == True) & (pl.col("route_of_targeted_receiver").is_not_null()))
              .then(pl.col("route_of_targeted_receiver"))
            # Otherwise, use route_ran
              .otherwise(pl.col("route_ran"))
        )
    )
    # -----------------------
    # Only Keep Routes Of Interest
    # -----------------------
    .filter(pl.col("route").is_in(["HITCH", "IN", "OUT", "GO", "POST", "CORNER"]))
)

print(f"WR Route Data:")
print(f"    - Dimensions: {bdb26_df.shape}")
print(f"    - Number of Unique WRs: {bdb26_df.select("nfl_id").n_unique()}")
print(f"    - Number of Player-Plays: {bdb26_df.select("player_play_id").n_unique()}")
bdb26_df

In [ ]:
# ==========================================
# WR & Assigned Defender Player-Tracking Data
# ==========================================

wr_separation_df = (
    pl.read_csv(str(SEPARATION_DATA_DIR / "01_2023_Rookie_Player_Time_Level_Separation_Data.csv"))
)

# -------------------------------
# For BDB2027 Route Data, Use route_ran
# -------------------------------

bdb2027_extra_route_data = (
    bdb26_df
    .filter( 
        pl.col("route_ran").is_not_null()
    )
    .group_by("player_play_id")
    .agg(
        route_ran = pl.col("route_ran").first()
    )
)

# -------------------------------
# For BDB2026 Route Data For Targeted 
# Receivers, Use route_of_targetered_receiver
# -------------------------------

target_routes = (
    bdb26_df
    .filter(
        pl.col("target"), 
        pl.col("route_of_targeted_receiver").is_not_null()
    )
    .group_by("player_play_id")
    .agg(
        route_of_targeted_receiver = pl.col("route_of_targeted_receiver").first()
    )
)

# -------------------------------
# Target Data
# -------------------------------

target_data = (
    bdb26_df
    .select("player_play_id", "target")
    .unique()
)

# -------------------------------
# Add Routes To WR Separation Data
# -------------------------------

wr_separation_df = (
    wr_separation_df
    # Add target data
    .join(
        target_data, 
        on = "player_play_id", 
        how = "inner"
    )
    # Add non-targeted routes
    .join(
        bdb2027_extra_route_data, 
        on = "player_play_id", 
        how = "left"
    )
    # Add targeted routes
    .join(
        target_routes, 
        on = "player_play_id", 
        how = "left"
    )
    # Create route variable
    .with_columns(
        route = (
            # If target and has route_of_targeted_receiver data, then use that
            pl.when((pl.col("target") == True) & (pl.col("route_of_targeted_receiver").is_not_null()))
              .then(pl.col("route_of_targeted_receiver"))
            # Otherwise, use route_ran
              .otherwise(pl.col("route_ran"))
        )
    )
)

# -----------------------------
# Only Consider These Route Types
# -----------------------------

wr_separation_df = (
    wr_separation_df
    .filter(pl.col("route").is_in(["HITCH", "IN", "OUT", "GO", "POST", "CORNER"]))
)


print(f"WR & Assigned Defender Data:")
print(f"    - Dimensions: {wr_separation_df.shape}")
print(f"    - Number of Unique WRs: {wr_separation_df.select("wr_nfl_id").n_unique()}")
print(f"    - Number of Player-Plays: {wr_separation_df.select("player_play_id").n_unique()}")
wr_separation_df

<br><br/>
## Flag Cutting Routes
***

In [ ]:
# ==========================================
# Configuration
# ==========================================

# # Time between frames -- 0.1 seconds
# DT = 0.1

# # Add smoothing window to reduce single-frame noise
# SMOOTH_WINDOW = 3
# # Add heading-change window to detect meaningful turns
# HEADING_WINDOW = 3       # 3 frames = 0.3 seconds
# # Post-cut observation time. Allow the WR to continue the route
# POST_WINDOW = 5          # 5 frames = 0.5 seconds

# # Minimum cut angle. Rejects minor direction changes
# MIN_CUT_ANGLE = 35.0
# # Minimum pre-cut speed. Rejects non-meaningful speeds
# MIN_PRE_CUT_SPEED = 3.0
# # Minumum post cut distance. Rejects cuts with little continuation.
# MIN_POST_CUT_DISPLACEMENT = 1.5

# # Routes we're interesting in
# CUT_ROUTES = ["HITCH", "IN", "OUT"]

DT = 0.1
SMOOTH_WINDOW = 3
HEADING_WINDOW = 2

MIN_CUT_ANGLE = 20.0
MIN_PRE_CUT_SPEED = 2.0

CUT_ROUTES = ["HITCH", "IN", "OUT"]

# ==========================================
# 1. Create one metadata row per WR route
# ==========================================

route_metadata = (
    wr_separation_df
    .select([
        "game_play_id",
        "player_play_id",
        "wr_nfl_id",
        "wr_name",
        "target",
        "route",
        "assigned_defender_nfl_id",
        "assigned_defender_name",
        "assigned_defender_position",
        "assignment_score",
    ])
    .unique(subset=["player_play_id"])
)

# Keep only the three cutting routes for this analysis.
route_metadata = route_metadata.filter(
    pl.col("route").is_in(CUT_ROUTES)
)

# Check that route metadata is unique.
assert (
    route_metadata
    .group_by("player_play_id")
    .len()
    .filter(pl.col("len") > 1)
    .height == 0
), "A player_play_id has multiple route metadata rows."

# ==========================================
# 2. Retrieve the WR's frame-level tracking
# ==========================================

# Join using player_play_id AND nfl_id.
# This protects against accidentally joining a different
# player's trajectory from the same play.

wr_tracking = (
    bdb26_df
    .select([
        "player_play_id",
        "nfl_id",
        "frame_id",
        "x",
        "y",
        "s",
        "dir",
        "o",
    ])
    .join(
        route_metadata.select([
            "player_play_id",
            "wr_nfl_id",
        ]),
        left_on=["player_play_id", "nfl_id"],
        right_on=["player_play_id", "wr_nfl_id"],
        how="inner",
    )
    .sort(["player_play_id", "frame_id"])
)

# ==========================================
# 3. Retrieve the assigned defender's tracking
# ==========================================

defender_tracking = (
    wr_separation_df
    .select([
        "player_play_id",
        "frame_id",
        "defender_x",
        "defender_y",
        "defender_dir",
        "defender_o",
        "wr_cb_distance",
    ])
    .unique(subset=["player_play_id", "frame_id"])
    .sort("player_play_id", "frame_id")
)

# At this point:
# - wr_tracking has the WR's movement
# - defender_tracking has the assigned DB's movement
# - route_metadata has the route label and target status

In [ ]:
# ==========================================
# Helper functions
# ==========================================

def smooth_1d(values, window=3):
    """
    Centered moving average with edge handling.
    Preserves the length of the input array.
    """
    values = np.asarray(values, dtype=float)

    if len(values) < 3:
        return values.copy()

    out = np.empty_like(values)

    half = window // 2

    for i in range(len(values)):
        start = max(0, i - half)
        end = min(len(values), i + half + 1)

        window_values = values[start:end]
        finite_values = window_values[np.isfinite(window_values)]

        out[i] = (
            finite_values.mean()
            if len(finite_values) > 0
            else np.nan
        )

    return out


def angle_difference_deg(angle1, angle2):
    """
    Smallest absolute angular difference in degrees.
    Result is between 0 and 180.
    """
    return abs((angle2 - angle1 + 180) % 360 - 180)


def bearing_deg(dx, dy):
    """
    Direction of travel from coordinate displacement.
    Uses the standard mathematical convention:
    0 degrees = +x, 90 degrees = +y.
    """
    return np.degrees(np.arctan2(dy, dx)) % 360


def trajectory_heading_change(x, y, i, window=3):
    """
    Compare the average path direction immediately before
    and after index i.

    Returns:
        heading change in degrees
    """
    n = len(x)

    if i - window < 0 or i + window >= n:
        return np.nan

    pre_dx = x[i] - x[i - window]
    pre_dy = y[i] - y[i - window]

    post_dx = x[i + window] - x[i]
    post_dy = y[i + window] - y[i]

    # If either displacement is almost zero, heading is unreliable.
    if np.hypot(pre_dx, pre_dy) < 0.15:
        return np.nan

    if np.hypot(post_dx, post_dy) < 0.15:
        return np.nan

    pre_heading = bearing_deg(pre_dx, pre_dy)
    post_heading = bearing_deg(post_dx, post_dy)

    return angle_difference_deg(pre_heading, post_heading)


def route_trajectory_features(
    frame_id,
    x,
    y,
    tracking_speed,
    route,
):
    """
    Identify the strongest candidate cut and calculate
    movement features for one WR route.
    """

    frame_id = np.asarray(frame_id, dtype=int)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    tracking_speed = np.asarray(tracking_speed, dtype=float)

    # Ensure chronological order.
    order = np.argsort(frame_id)

    frame_id = frame_id[order]
    x = x[order]
    y = y[order]
    tracking_speed = tracking_speed[order]

    n = len(frame_id)

    result = {
        "n_frames": n,
        "route_duration_sec": (
            (frame_id[-1] - frame_id[0]) * DT
            if n > 1 else 0.0
        ),
        "cut_frame_id": None,
        "cut_angle_deg": np.nan,
        "pre_cut_speed": np.nan,
        "min_pre_cut_accel": np.nan,
        "post_cut_displacement": np.nan,
        "post_cut_speed": np.nan,
        "cut_event_detected": False,
        "sufficient_post_cut_data": False,
    }

    # Need enough observations on both sides of a candidate cut.
    minimum_frames = 2 * HEADING_WINDOW + 2

    if n < minimum_frames:
        return result

    # Smooth coordinates to reduce frame-to-frame noise.
    x_smooth = smooth_1d(x, SMOOTH_WINDOW)
    y_smooth = smooth_1d(y, SMOOTH_WINDOW)

    # Candidate cut indices must have a full window before and after.
    candidates = []

    for i in range(HEADING_WINDOW, n - HEADING_WINDOW):
        angle = trajectory_heading_change(
            x_smooth,
            y_smooth,
            i,
            window=HEADING_WINDOW,
        )

        if np.isfinite(angle):
            candidates.append((i, angle))

    if not candidates:
        return result

    # Identify the largest change in direction.
    # We will later apply the route-specific cut criteria.
    event_i, event_angle = max(
        candidates,
        key=lambda item: item[1],
    )

    result["cut_frame_id"] = int(frame_id[event_i])
    result["cut_angle_deg"] = float(event_angle)

    # Pre-cut speed: average tracking speed during the
    # 0.3 seconds immediately preceding the candidate event.
    pre_start = max(0, event_i - HEADING_WINDOW)

    pre_speeds = tracking_speed[pre_start:event_i]
    pre_speeds = pre_speeds[np.isfinite(pre_speeds)]

    if len(pre_speeds):
        result["pre_cut_speed"] = float(pre_speeds.mean())

    # Acceleration estimated from the change in tracking speed.
    speed_accel = np.diff(tracking_speed) / DT

    # Find braking during the 0.3 seconds leading up to the event.
    accel_start = max(0, event_i - HEADING_WINDOW)

    pre_accels = speed_accel[accel_start:event_i]
    pre_accels = pre_accels[np.isfinite(pre_accels)]

    if len(pre_accels):
        result["min_pre_cut_accel"] = float(pre_accels.min())

    # Require a full 0.5-second post-event window to calculate
    # post-cut displacement and speed.
    if event_i + POST_WINDOW < n:
        post_dx = (
            x_smooth[event_i + POST_WINDOW]
            - x_smooth[event_i]
        )
        post_dy = (
            y_smooth[event_i + POST_WINDOW]
            - y_smooth[event_i]
        )

        result["post_cut_displacement"] = float(
            np.hypot(post_dx, post_dy)
        )

        post_speeds = tracking_speed[
            event_i + 1:event_i + POST_WINDOW + 1
        ]
        post_speeds = post_speeds[np.isfinite(post_speeds)]

        if len(post_speeds):
            result["post_cut_speed"] = float(post_speeds.mean())

        result["sufficient_post_cut_data"] = True

    # The event itself is only a candidate at this stage.
    result["cut_event_detected"] = (
        event_angle >= MIN_CUT_ANGLE
    )

    return result

In [ ]:
# ==========================================
# 1. Combine WR, Defender, And Route Information
# ==========================================

route_tracking = (
    wr_tracking
    .join(
        defender_tracking,
        on=["player_play_id", "frame_id"],
        how="left",
    )
    .join(
        route_metadata.select([
            "player_play_id",
            "game_play_id",
            "target",
            "route",
            "wr_name",
            "assigned_defender_nfl_id",
            "assigned_defender_name",
            "assigned_defender_position",
            "assignment_score",
        ]),
        on="player_play_id",
        how="left",
        suffix="_meta",
    )
    .sort(["player_play_id", "frame_id"])
)

# ==========================================
# 2. Evaluate every untargeted route
# ==========================================

untargeted_tracking = route_tracking.filter(
    pl.col("target") == False
)

route_results = []

for key, group in untargeted_tracking.partition_by(
    "player_play_id",
    as_dict=True,
).items():

    player_play_id = (
        key[0] if isinstance(key, tuple) else key
    )

    group = group.sort("frame_id")

    route = group["route"][0]

    # WR trajectory features
    features = route_trajectory_features(
        frame_id=group["frame_id"].to_numpy(),
        x=group["x"].to_numpy(),
        y=group["y"].to_numpy(),
        tracking_speed=group["s"].to_numpy(),
        route=route,
    )

    # --------------------------------------
    # Defender response at the WR's event
    # --------------------------------------

    defender_angle_change = np.nan

    event_frame = features["cut_frame_id"]

    if event_frame is not None:
        event_rows = group.filter(
            pl.col("frame_id") == event_frame
        )

        if event_rows.height == 1:
            event_i = group["frame_id"].to_list().index(
                event_frame
            )

            defender_x = group["defender_x"].to_numpy()
            defender_y = group["defender_y"].to_numpy()

            if (
                np.isfinite(defender_x).all()
                and np.isfinite(defender_y).all()
            ):
                defender_x_smooth = smooth_1d(
                    defender_x,
                    SMOOTH_WINDOW,
                )
                defender_y_smooth = smooth_1d(
                    defender_y,
                    SMOOTH_WINDOW,
                )

                defender_angle_change = (
                    trajectory_heading_change(
                        defender_x_smooth,
                        defender_y_smooth,
                        event_i,
                        window=HEADING_WINDOW,
                    )
                )

    features["defender_heading_change_deg"] = (
        defender_angle_change
    )

    # --------------------------------------
    # Route-specific inclusion criteria
    # --------------------------------------

    enough_pre_speed = (
        np.isfinite(features["pre_cut_speed"])
        and features["pre_cut_speed"] >= MIN_PRE_CUT_SPEED
    )

    meaningful_cut = (
        np.isfinite(features["cut_angle_deg"])
        and features["cut_angle_deg"] >= MIN_CUT_ANGLE
    )

    enough_post_movement = (
        features["sufficient_post_cut_data"]
        and np.isfinite(features["post_cut_displacement"])
        and features["post_cut_displacement"]
            >= MIN_POST_CUT_DISPLACEMENT
    )

    braking_pass = (
        np.isfinite(features["min_pre_cut_accel"])
        and features["min_pre_cut_accel"] <= -1.5
    )

    # Apply separate rules by route.
    if route == "HITCH":
        route_execution_pass = (
            meaningful_cut
            and enough_pre_speed
            and braking_pass
            and enough_post_movement
        )

    elif route in ["IN", "OUT"]:
        route_execution_pass = (
            meaningful_cut
            and enough_pre_speed
            and enough_post_movement
        )

    else:
        route_execution_pass = False

    # --------------------------------------
    # Classification status
    # --------------------------------------

    enough_tracking = (
        features["n_frames"] >= 2 * HEADING_WINDOW + 2
        and features["sufficient_post_cut_data"]
    )

    if not enough_tracking:
        status = "INDETERMINATE"

    elif route_execution_pass:
        status = "CREDIBLE"

    else:
        status = "NOT_CREDIBLE"

    features["route"] = route
    features["player_play_id"] = player_play_id
    features["route_execution_pass"] = route_execution_pass
    features["braking_pass"] = braking_pass
    features["meaningful_cut"] = meaningful_cut
    features["enough_pre_speed"] = enough_pre_speed
    features["enough_post_movement"] = enough_post_movement
    features["classification_status"] = status

    route_results.append(features)

# Convert the route-level results back to Polars.
cut_route_features = pl.DataFrame(route_results)

In [ ]:
# ==========================================
# Final route-level classification
# ==========================================

# Keep one row per player-play.
final_cut_routes = (
    route_metadata
    .join(
        cut_route_features,
        on=["player_play_id", "route"],
        how="left",
    )
    .with_columns(
        pl.when(pl.col("target") == True)
        .then(pl.lit("TARGETED_AUTO_INCLUDE"))
        .otherwise(pl.col("classification_status"))
        .alias("classification_status")
    )
    .with_columns(
        pl.when(pl.col("target") == True)
        .then(pl.lit(True))
        .otherwise(
            pl.col("classification_status") == "CREDIBLE"
        )
        .alias("credible_separation_attempt")
    )
)

# Inspect the results.
print(
    final_cut_routes
    .group_by(["route", "target", "classification_status"])
    .len()
    .sort(["route", "target", "classification_status"])
)

# Routes retained for the separation analysis.
credible_cut_routes = final_cut_routes.filter(
    pl.col("credible_separation_attempt")
)

# Routes retained that were originally untargeted.
credible_untargeted_cut_routes = credible_cut_routes.filter(
    pl.col("target") == False
)

In [ ]:
final_cut_routes.filter(pl.col("target") == False, pl.col("classification_status") == "INDETERMINATE")

In [ ]:
untargeted_summary = (
    final_cut_routes
    .filter(pl.col("target") == False)
    .group_by(["route", "classification_status"])
    .len()
    .sort(["route", "classification_status"])
)

untargeted_summary

In [ ]:
feature_summary = (
    final_cut_routes
    .filter(pl.col("target") == False)
    .group_by("route")
    .agg([
        pl.len().alias("n_routes"),
        pl.col("cut_angle_deg").median().alias("median_cut_angle"),
        pl.col("pre_cut_speed").median().alias("median_pre_cut_speed"),
        pl.col("min_pre_cut_accel").median().alias("median_pre_cut_accel"),
        pl.col("post_cut_displacement").median().alias("median_post_cut_displacement"),
        pl.col("defender_heading_change_deg").median().alias("median_defender_heading_change"),
    ])
)

print(feature_summary)

In [ ]:
audit_routes = (
    final_cut_routes
    .filter(pl.col("target") == False)
    .select([
        "player_play_id",
        "wr_name",
        "route",
        "classification_status",
        "cut_frame_id",
        "cut_angle_deg",
        "pre_cut_speed",
        "min_pre_cut_accel",
        "post_cut_displacement",
        "defender_heading_change_deg",
        "meaningful_cut",
        "braking_pass",
        "enough_pre_speed",
        "enough_post_movement",
    ])
    .sort(
        ["classification_status", "route", "cut_angle_deg"],
        descending=[False, False, True],
    )
)

audit_routes.head(50)

In [ ]:
# Count which criteria fail for each route type.
# This uses the existing Boolean flags from the classification code.

failure_summary = (
    final_cut_routes
    .filter(pl.col("target") == False)
    .group_by("route")
    .agg([
        pl.len().alias("n_routes"),

        (pl.col("classification_status") == "CREDIBLE")
        .sum().alias("n_credible"),

        (pl.col("classification_status") == "INDETERMINATE")
        .sum().alias("n_indeterminate"),

        (~pl.col("meaningful_cut").fill_null(False))
        .sum().alias("n_fail_cut_angle"),

        (~pl.col("enough_pre_speed").fill_null(False))
        .sum().alias("n_fail_pre_speed"),

        (~pl.col("enough_post_movement").fill_null(False))
        .sum().alias("n_fail_post_movement"),

        (~pl.col("braking_pass").fill_null(False))
        .sum().alias("n_fail_braking"),

        pl.col("cut_angle_deg").median().alias("median_cut_angle"),
        pl.col("pre_cut_speed").median().alias("median_pre_cut_speed"),
        pl.col("post_cut_displacement").median().alias(
            "median_post_cut_displacement"
        ),
    ])
    .sort("route")
)

failure_summary

In [ ]:
indeterminate_audit = (
    final_cut_routes
    .filter(
        (pl.col("target") == False)
        & (pl.col("classification_status") == "INDETERMINATE")
    )
    .select([
        "player_play_id",
        "wr_name",
        "route",
        "n_frames",
        "route_duration_sec",
        "cut_frame_id",
        "cut_angle_deg",
        "sufficient_post_cut_data",
        "post_cut_displacement",
    ])
    .sort(["route", "n_frames"])
)

indeterminate_audit.group_by("sufficient_post_cut_data").agg(pl.len())

# ------------------------------------------------------------

In [ ]:
import numpy as np
import polars as pl

# ============================================================
# PARAMETERS
# ============================================================

DT = 0.1

SMOOTH_WINDOW = 3
HEADING_WINDOW = 2

POST_WINDOW = 5  # Diagnostic only; not required for inclusion

MIN_CUT_ANGLE = 20.0
MIN_PRE_CUT_SPEED = 2.0

MIN_POST_CUT_DISPLACEMENT = 1.5  # Diagnostic only

CUT_ROUTES = ["HITCH", "IN", "OUT"]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def smooth_1d(values, window=3):
    """Smooth a 1D array using a centered moving average."""
    values = np.asarray(values, dtype=float)

    if len(values) < 3 or window <= 1:
        return values.copy()

    out = np.empty_like(values)
    half = window // 2

    for i in range(len(values)):
        start = max(0, i - half)
        end = min(len(values), i + half + 1)

        window_values = values[start:end]
        finite_values = window_values[np.isfinite(window_values)]

        out[i] = (
            finite_values.mean()
            if len(finite_values) > 0
            else np.nan
        )

    return out


def angle_difference_deg(angle1, angle2):
    """Smallest absolute angular difference in degrees."""
    return abs((angle2 - angle1 + 180) % 360 - 180)


def bearing_deg(dx, dy):
    """Calculate the direction of a displacement vector."""
    return np.degrees(np.arctan2(dy, dx)) % 360


def trajectory_heading_change(x, y, i, window=2):
    """
    Compare movement heading before and after frame i.

    Each heading is calculated from the displacement over
    `window` frames.
    """
    n = len(x)

    if i - window < 0 or i + window >= n:
        return np.nan

    pre_dx = x[i] - x[i - window]
    pre_dy = y[i] - y[i - window]

    post_dx = x[i + window] - x[i]
    post_dy = y[i + window] - y[i]

    # Avoid calculating heading from nearly stationary movement.
    if (
        not np.all(np.isfinite([pre_dx, pre_dy, post_dx, post_dy]))
        or np.hypot(pre_dx, pre_dy) < 0.15
        or np.hypot(post_dx, post_dy) < 0.15
    ):
        return np.nan

    pre_heading = bearing_deg(pre_dx, pre_dy)
    post_heading = bearing_deg(post_dx, post_dy)

    return angle_difference_deg(pre_heading, post_heading)


def route_trajectory_features(
    frame_id,
    x,
    y,
    tracking_speed,
    route,
):
    """
    Evaluate candidate turning points throughout a route.

    A route passes the movement criteria if at least one
    candidate has:
        1. Heading change >= MIN_CUT_ANGLE
        2. Pre-event speed >= MIN_PRE_CUT_SPEED

    Post-event displacement and braking are diagnostic only.
    """

    frame_id = np.asarray(frame_id, dtype=int)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    tracking_speed = np.asarray(tracking_speed, dtype=float)

    # Sort all arrays by frame.
    order = np.argsort(frame_id)

    frame_id = frame_id[order]
    x = x[order]
    y = y[order]
    tracking_speed = tracking_speed[order]

    n = len(frame_id)

    result = {
        "n_frames": n,
        "route_duration_sec": (
            (frame_id[-1] - frame_id[0]) * DT
            if n > 1 else 0.0
        ),
        "cut_frame_id": None,
        "cut_angle_deg": np.nan,
        "pre_cut_speed": np.nan,
        "min_pre_cut_accel": np.nan,
        "post_cut_displacement": np.nan,
        "post_cut_speed": np.nan,
        "sufficient_post_cut_data": False,
        "has_evaluable_turn": False,
        "qualifying_cut_event": False,
    }

    # Need enough observations to calculate headings on both sides.
    minimum_frames = 2 * HEADING_WINDOW + 1

    if n < minimum_frames:
        return result

    x_smooth = smooth_1d(x, SMOOTH_WINDOW)
    y_smooth = smooth_1d(y, SMOOTH_WINDOW)

    # Speed-derived acceleration is diagnostic.
    speed_accel = np.diff(tracking_speed) / DT

    # --------------------------------------------------------
    # Evaluate every candidate turning point.
    # --------------------------------------------------------

    candidates = []

    for i in range(HEADING_WINDOW, n - HEADING_WINDOW):

        angle = trajectory_heading_change(
            x_smooth,
            y_smooth,
            i,
            window=HEADING_WINDOW,
        )

        if not np.isfinite(angle):
            continue

        # Mean speed over the frames immediately before the event.
        pre_start = max(0, i - HEADING_WINDOW)

        pre_speeds = tracking_speed[pre_start:i]
        pre_speeds = pre_speeds[np.isfinite(pre_speeds)]

        pre_speed = (
            float(pre_speeds.mean())
            if len(pre_speeds) > 0
            else np.nan
        )

        # Minimum speed-derived acceleration before the event.
        pre_accels = speed_accel[pre_start:i]
        pre_accels = pre_accels[np.isfinite(pre_accels)]

        min_pre_accel = (
            float(pre_accels.min())
            if len(pre_accels) > 0
            else np.nan
        )

        candidate = {
            "i": i,
            "frame_id": int(frame_id[i]),
            "angle": float(angle),
            "pre_speed": pre_speed,
            "min_pre_accel": min_pre_accel,
        }

        candidates.append(candidate)

    if not candidates:
        return result

    result["has_evaluable_turn"] = True

    # --------------------------------------------------------
    # Prefer the strongest candidate that passes BOTH criteria.
    #
    # If none passes, retain the strongest turn for diagnostics.
    # --------------------------------------------------------

    qualifying_candidates = [
        c for c in candidates
        if (
            c["angle"] >= MIN_CUT_ANGLE
            and np.isfinite(c["pre_speed"])
            and c["pre_speed"] >= MIN_PRE_CUT_SPEED
        )
    ]

    if qualifying_candidates:
        selected = max(
            qualifying_candidates,
            key=lambda c: c["angle"],
        )
        result["qualifying_cut_event"] = True

    else:
        selected = max(
            candidates,
            key=lambda c: c["angle"],
        )

    event_i = selected["i"]

    result["cut_frame_id"] = selected["frame_id"]
    result["cut_angle_deg"] = selected["angle"]
    result["pre_cut_speed"] = selected["pre_speed"]
    result["min_pre_cut_accel"] = selected["min_pre_accel"]

    # --------------------------------------------------------
    # Post-event metrics are diagnostics only.
    # These do NOT determine route inclusion.
    # --------------------------------------------------------

    if event_i + POST_WINDOW < n:

        post_dx = (
            x_smooth[event_i + POST_WINDOW]
            - x_smooth[event_i]
        )

        post_dy = (
            y_smooth[event_i + POST_WINDOW]
            - y_smooth[event_i]
        )

        if np.isfinite([post_dx, post_dy]).all():

            result["post_cut_displacement"] = float(
                np.hypot(post_dx, post_dy)
            )

            post_speeds = tracking_speed[
                event_i + 1:event_i + POST_WINDOW + 1
            ]
            post_speeds = post_speeds[
                np.isfinite(post_speeds)
            ]

            if len(post_speeds) > 0:
                result["post_cut_speed"] = float(
                    post_speeds.mean()
                )

            result["sufficient_post_cut_data"] = True

    return result

In [ ]:
# ============================================================
# ROUTE METADATA: ONE ROW PER PLAYER-PLAY
# ============================================================

route_metadata = (
    wr_separation_df
    .select([
        "game_play_id",
        "player_play_id",
        "wr_nfl_id",
        "wr_name",
        "target",
        "route",
        "assigned_defender_nfl_id",
        "assigned_defender_name",
        "assigned_defender_position",
        "assignment_score",
    ])
    .unique(
        subset=["player_play_id"],
        keep="first",
        maintain_order=True,
    )
    .filter(pl.col("route").is_in(CUT_ROUTES))
)


# ============================================================
# RECEIVER TRACKING
# ============================================================

# Join by player-play, then explicitly retain the receiver's
# own tracking rows using nfl_id == wr_nfl_id.

wr_tracking = (
    bdb26_df
    .select([
        "player_play_id",
        "nfl_id",
        "frame_id",
        "x",
        "y",
        "s",
        "dir",
        "o",
    ])
    .join(
        route_metadata.select([
            "player_play_id",
            "wr_nfl_id",
        ]),
        on="player_play_id",
        how="inner",
    )
    .filter(pl.col("nfl_id") == pl.col("wr_nfl_id"))
    .drop("wr_nfl_id")
    .sort(["player_play_id", "frame_id"])
)


# ============================================================
# ASSIGNED DEFENDER TRACKING
# ============================================================

defender_tracking = (
    wr_separation_df
    .select([
        "player_play_id",
        "frame_id",
        "defender_x",
        "defender_y",
        "defender_dir",
        "defender_o",
        "wr_cb_distance",
    ])
    .unique(
        subset=["player_play_id", "frame_id"],
        keep="first",
        maintain_order=True,
    )
)


# ============================================================
# COMBINED TRACKING DATA
# ============================================================

route_tracking = (
    wr_tracking
    .join(
        defender_tracking,
        on=["player_play_id", "frame_id"],
        how="left",
    )
    .join(
        route_metadata.select([
            "player_play_id",
            "game_play_id",
            "wr_name",
            "target",
            "route",
            "assigned_defender_nfl_id",
            "assigned_defender_name",
            "assigned_defender_position",
            "assignment_score",
        ]),
        on="player_play_id",
        how="left",
    )
    .sort(["player_play_id", "frame_id"])
)

In [ ]:
# ============================================================
# CLASSIFY UNTARGETED ROUTES
# ============================================================

untargeted_tracking = route_tracking.filter(
    pl.col("target") == False
)

route_results = []

for key, group in untargeted_tracking.partition_by(
    "player_play_id",
    as_dict=True,
).items():

    player_play_id = key[0] if isinstance(key, tuple) else key

    group = group.sort("frame_id")
    route = group["route"][0]

    features = route_trajectory_features(
        frame_id=group["frame_id"].to_numpy(),
        x=group["x"].cast(pl.Float64).fill_null(float("nan")).to_numpy(),
        y=group["y"].cast(pl.Float64).fill_null(float("nan")).to_numpy(),
        tracking_speed=(
            group["s"]
            .cast(pl.Float64)
            .fill_null(float("nan"))
            .to_numpy()
        ),
        route=route,
    )

    # --------------------------------------------------------
    # Diagnostic: defender heading change around the selected
    # receiver event, where the tracking data are available.
    # --------------------------------------------------------

    defender_angle_change = np.nan
    event_frame = features["cut_frame_id"]

    if event_frame is not None:

        frame_values = group["frame_id"].to_list()

        if event_frame in frame_values:

            event_i = frame_values.index(event_frame)

            defender_x = (
                group["defender_x"]
                .cast(pl.Float64)
                .fill_null(float("nan"))
                .to_numpy()
            )

            defender_y = (
                group["defender_y"]
                .cast(pl.Float64)
                .fill_null(float("nan"))
                .to_numpy()
            )

            if len(defender_x) == len(frame_values):

                defender_x_smooth = smooth_1d(
                    defender_x,
                    SMOOTH_WINDOW,
                )

                defender_y_smooth = smooth_1d(
                    defender_y,
                    SMOOTH_WINDOW,
                )

                defender_angle_change = trajectory_heading_change(
                    defender_x_smooth,
                    defender_y_smooth,
                    event_i,
                    window=HEADING_WINDOW,
                )

    # --------------------------------------------------------
    # Classification
    # --------------------------------------------------------

    enough_tracking = (
        features["n_frames"] >= 2 * HEADING_WINDOW + 1
        and features["has_evaluable_turn"]
    )

    meaningful_cut = (
        np.isfinite(features["cut_angle_deg"])
        and features["cut_angle_deg"] >= MIN_CUT_ANGLE
    )

    enough_pre_speed = (
        np.isfinite(features["pre_cut_speed"])
        and features["pre_cut_speed"] >= MIN_PRE_CUT_SPEED
    )

    # Braking is diagnostic only, not mandatory.
    braking_pass = (
        np.isfinite(features["min_pre_cut_accel"])
        and features["min_pre_cut_accel"] <= -1.5
    )

    # Post-cut movement is diagnostic only, not mandatory.
    enough_post_movement = (
        features["sufficient_post_cut_data"]
        and np.isfinite(features["post_cut_displacement"])
        and features["post_cut_displacement"]
        >= MIN_POST_CUT_DISPLACEMENT
    )

    route_execution_pass = features["qualifying_cut_event"]

    if not enough_tracking:
        status = "INDETERMINATE"

    elif route_execution_pass:
        status = "CREDIBLE"

    else:
        status = "NOT_CREDIBLE"

    # Add identifiers and diagnostic flags.
    features["player_play_id"] = player_play_id
    features["route"] = route
    features["classification_status"] = status
    features["route_execution_pass"] = route_execution_pass
    features["meaningful_cut"] = meaningful_cut
    features["enough_pre_speed"] = enough_pre_speed
    features["braking_pass"] = braking_pass
    features["enough_post_movement"] = enough_post_movement
    features["defender_heading_change_deg"] = defender_angle_change

    route_results.append(features)


# Create the route-level feature dataframe.
if route_results:
    cut_route_features = pl.DataFrame(route_results)
else:
    cut_route_features = pl.DataFrame()

In [ ]:
# ============================================================
# FINAL ROUTE-LEVEL DATASET
# ============================================================

final_cut_routes = (
    route_metadata
    .join(
        cut_route_features,
        on=["player_play_id", "route"],
        how="left",
    )
    .with_columns([
        pl.when(pl.col("target") == True)
        .then(pl.lit("TARGETED_AUTO_INCLUDE"))
        .otherwise(pl.col("classification_status"))
        .alias("classification_status"),

        pl.when(pl.col("target") == True)
        .then(pl.lit(True))
        .otherwise(
            pl.col("classification_status") == "CREDIBLE"
        )
        .alias("credible_separation_attempt"),
    ])
)

# All credible routes, including targeted routes.
credible_cut_routes = final_cut_routes.filter(
    pl.col("credible_separation_attempt")
)

# Only credible untargeted routes.
credible_untargeted_cut_routes = credible_cut_routes.filter(
    pl.col("target") == False
)

In [ ]:
untargeted_summary = (
    final_cut_routes
    .filter(pl.col("target") == False)
    .group_by(["route", "classification_status"])
    .len()
    .sort(["route", "classification_status"])
)

print(untargeted_summary)

In [ ]:
feature_summary = (
    final_cut_routes
    .filter(pl.col("target") == False)
    .group_by("route")
    .agg([
        pl.len().alias("n_routes"),

        pl.col("cut_angle_deg")
        .median()
        .alias("median_cut_angle"),

        pl.col("pre_cut_speed")
        .median()
        .alias("median_pre_cut_speed"),

        pl.col("min_pre_cut_accel")
        .median()
        .alias("median_pre_cut_accel"),

        pl.col("post_cut_displacement")
        .median()
        .alias("median_post_cut_displacement"),

        pl.col("defender_heading_change_deg")
        .median()
        .alias("median_defender_heading_change"),
    ])
    .sort("route")
)

feature_summary

In [ ]:
credible_audit = (
    final_cut_routes
    .filter(pl.col("target") == False)
    .filter(pl.col("classification_status") == "CREDIBLE")
    .select([
        "player_play_id",
        "wr_name",
        "route",
        "n_frames",
        "cut_frame_id",
        "cut_angle_deg",
        "pre_cut_speed",
        "min_pre_cut_accel",
        "post_cut_displacement",
        "defender_heading_change_deg",
        "sufficient_post_cut_data",
    ])
    .sort(["route", "cut_angle_deg"], descending=[False, True])
)

print(credible_audit.select("player_play_id").to_series().to_list())

In [ ]:
audit_routes = (
    final_cut_routes
    .filter(pl.col("target") == False)
    .select([
        "player_play_id",
        "wr_name",
        "route",
        "classification_status",
        "n_frames",
        "route_duration_sec",
        "cut_frame_id",
        "cut_angle_deg",
        "pre_cut_speed",
        "min_pre_cut_accel",
        "post_cut_displacement",
        "sufficient_post_cut_data",
        "meaningful_cut",
        "enough_pre_speed",
        "braking_pass",
        "enough_post_movement",
    ])
    .sort(["classification_status", "route", "cut_angle_deg"])
    .filter(pl.col("classification_status") == "CREDIBLE")
)

audit_routes.head(100)